# ICS 3202: Artificial Intelligence — Practical Exercise 4
## Knowledge Engineering: Autonomous Self-Driving Trailer Agent

* **Student / Group Lead:** Rocliff Kadaji (`166133`)
* **Group Members:** *(Add remaining group member names and admission numbers here)*
* **Repository:** `KnowledgeEngineering` | **Notebook:** `Autonomous_Trailer.ipynb`
* **Adapted from:** *Smart Traffic Light Navigation Agent (Zainabu Muti)*

**Step 1: Identify the Task**

The task is to build a **Knowledge-Based Agent (KBA)** for an autonomous heavy-duty freight trailer navigating Kenya's highway network (from Practical Exercise 3) between **Nairobi** and **Kisumu**.

**Goal:** Make safe, legally compliant driving and routing decisions with minimal collision risk and optimal path cost.

**Step 2: Assemble Relevant Knowledge**

The system requires:
* **Regional Highway Directed Graph & Distances (from Practical Exercise 3):** Waypoints across Nairobi, Central Highlands, Rift Valley, and Lake Victoria Basin, road distances $g(n)$ in km, and straight-line heuristic distances $h(n)$.
* **Trailer & Road Constraints:** Cargo load (tonnes), maximum axle/bridge weight limits per highway segment, and road closure/congestion status.
* **Real-Time Weather Percepts (External API):** Live weather conditions fetched via the public **Open-Meteo Weather API** (`https://api.open-meteo.com/v1/forecast`, no API key required).
* **Domain Safety Rules:** Mapping `(RoadStatus, Weather)` percepts to safe actions and speed limits.

**Step 3: Define the Vocabulary**

* **Constants (Waypoints from PE3):** `Nairobi`, `Naivasha`, `Thika`, `Nyeri`, `Nyahururu`, `Gilgil`, `Nakuru`, `Mau_Summit`, `Kericho`, `Eldoret`, `Ahero`, `Kisumu`
* **Constants (States & Actions):**
  * *Weather:* `Clear`, `Rain`, `Fog`, `Storm`
  * *RoadStatus:* `Open`, `Congested`, `Blocked`, `WeightRestricted`
  * *Actions:* `Proceed_Normal`, `Reduce_Speed_Increase_Headway`, `Reroute_Alternative`, `Halt_At_Safe_Bay`
* **Predicates (Encoded in Python KB structures & inference methods):**
  * `Connected(u, v, distance_km)`
  * `MaxAxleLoad(u, v, max_tonnes)`
  * `WeatherAt(town, condition)`
  * `RoadStatus(u, v, status)`
  * `CanTraverse(trailer, u, v)`

**Step 4: Encode General Knowledge about the Domain** `[3 marks]`

Below we declare the vocabulary, encode the Practical Exercise 3 directed graph and distances, define the safety rules, and query the Open-Meteo API for live weather updates.

In [1]:
import heapq
import json
import math
import urllib.request

# 1. VOCABULARY: PE3 Highway Waypoints (Nairobi, Central Highlands, Rift Valley, Lake Victoria Basin)
WAYPOINTS = {
    "Nairobi":    {"coords": (-1.2864, 36.8172), "weather": "Clear"},
    "Naivasha":   {"coords": (-0.7172, 36.4310), "weather": "Clear"},
    "Thika":      {"coords": (-1.0332, 37.0693), "weather": "Clear"},
    "Nyeri":      {"coords": (-0.4167, 36.9500), "weather": "Clear"},
    "Nyahururu":  {"coords": (0.0381,  36.3634), "weather": "Clear"},
    "Gilgil":     {"coords": (-0.4958, 36.3173), "weather": "Clear"},
    "Nakuru":     {"coords": (-0.3031, 36.0800), "weather": "Clear"},
    "Mau_Summit": {"coords": (-0.1636, 35.6847), "weather": "Clear"},
    "Kericho":    {"coords": (-0.3677, 35.2831), "weather": "Clear"},
    "Eldoret":    {"coords": (0.5143,  35.2698), "weather": "Clear"},
    "Ahero":      {"coords": (-0.1744, 34.9167), "weather": "Clear"},
    "Kisumu":     {"coords": (-0.0917, 34.7680), "weather": "Clear"},
}

# 2. DIRECTED GRAPH & DISTANCES (Exact match to Practical Exercise 3)
# Format: origin -> [(destination, distance_km, max_weight_tonnes, road_status)]
DIRECTED_GRAPH = {
    "Nairobi":    [("Naivasha", 90, 50, "Open"), ("Thika", 45, 50, "Open")],
    "Thika":      [("Nyeri", 106, 48, "Open")],
    "Nyeri":      [("Nyahururu", 101, 45, "Open")],
    "Nyahururu":  [("Gilgil", 67, 45, "Open")],
    "Naivasha":   [("Gilgil", 28, 50, "Open")],
    "Gilgil":     [("Nakuru", 43, 50, "Open")],
    "Nakuru":     [("Mau_Summit", 52, 50, "Open")],
    "Mau_Summit": [("Kericho", 48, 48, "Open"), ("Eldoret", 103, 50, "Open")],
    "Kericho":    [("Ahero", 62, 48, "Open")],
    "Eldoret":    [("Kisumu", 118, 48, "Open")],
    "Ahero":      [("Kisumu", 22, 50, "Open")],
    "Kisumu":     []
}

# 3. DOMAIN SAFETY RULES
DOMAIN_RULES = {
    ("Open", "Clear"):      {"action": "Proceed_Normal", "max_speed_kmh": 80, "advice": "Safe highway conditions: maintain cruise speed (80 km/h cap)."},
    ("Open", "Rain"):       {"action": "Reduce_Speed_Increase_Headway", "max_speed_kmh": 55, "advice": "Wet tarmac: engage engine retarder and double following distance."},
    ("Open", "Fog"):        {"action": "Reduce_Speed_Increase_Headway", "max_speed_kmh": 40, "advice": "Rift Valley/highland fog: activate fog lamps and lane-keep assist."},
    ("Open", "Storm"):      {"action": "Halt_At_Safe_Bay", "max_speed_kmh": 0, "advice": "Severe storm: park at nearest designated truck lay-by."},
    ("Congested", "Clear"): {"action": "Reduce_Speed_Increase_Headway", "max_speed_kmh": 35, "advice": "Heavy traffic: downshift and maintain 50m safe gap."},
    ("Congested", "Rain"):  {"action": "Reduce_Speed_Increase_Headway", "max_speed_kmh": 25, "advice": "Wet congestion: high jackknife risk, crawl cautiously."},
    ("Blocked", "Any"):     {"action": "Reroute_Alternative", "max_speed_kmh": 0, "advice": "Road blocked: compute alternative path via search tree."},
}

# 4. HEURISTIC DISTANCE MEASURE h(n) (Haversine straight-line distance in km)
def haversine_km(town_a, town_b):
    lat1, lon1 = WAYPOINTS[town_a]["coords"]
    lat2, lon2 = WAYPOINTS[town_b]["coords"]
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return round(2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a)), 2)

# 5. EXTERNAL API CONNECTOR (Open-Meteo Live Weather)
def sync_weather_api(town):
    lat, lon = WAYPOINTS[town]["coords"]
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    try:
        with urllib.request.urlopen(url, timeout=5) as resp:
            code = json.loads(resp.read().decode())["current_weather"]["weathercode"]
            cond = "Clear" if code in [0, 1, 2, 3] else ("Fog" if code in [45, 48] else ("Rain" if code in [51, 53, 55, 61, 63, 65, 80, 81, 82] else "Storm"))
            WAYPOINTS[town]["weather"] = cond
            return f"API Sync [{town}]: Weather='{cond}' (WMO Code {code})"
    except Exception as e:
        return f"API Offline [{town}]: Using cached '{WAYPOINTS[town]['weather']}' ({e})"

print(sync_weather_api("Nakuru"))
print(sync_weather_api("Kericho"))

API Sync [Nakuru]: Weather='Rain' (WMO Code 51)
API Sync [Kericho]: Weather='Clear' (WMO Code 3)


**Step 5: Add New Facts to the Knowledge Base** `[4 marks]`

In [2]:
def show_kb():
    print("\n================ TRAILER KNOWLEDGE BASE ================")
    print("Waypoint Weather Percepts:")
    for town, info in WAYPOINTS.items():
        print(f"  {town:12} -> Weather: {info['weather']}")
    print("\nDirected Highway Edges:")
    for src, edges in DIRECTED_GRAPH.items():
        for dst, dist, max_w, status in edges:
            print(f"  {src:12} -> {dst:12} | {dist:3} km | MaxLoad: {max_w}t | Status: {status}")
    print("========================================================\n")

def add_or_update_edge(src, dst, dist_km, max_weight_t, status="Open"):
    if src not in DIRECTED_GRAPH:
        DIRECTED_GRAPH[src] = []
    for idx, (d, _, _, _) in enumerate(DIRECTED_GRAPH[src]):
        if d == dst:
            DIRECTED_GRAPH[src][idx] = (dst, dist_km, max_weight_t, status)
            print(f"Updated Edge Fact: {src} -> {dst} ({dist_km} km, {max_weight_t}t, {status})")
            return
    DIRECTED_GRAPH[src].append((dst, dist_km, max_weight_t, status))
    print(f"Added New Edge Fact: {src} -> {dst} ({dist_km} km, {max_weight_t}t, {status})")

def update_weather(town, condition):
    if town in WAYPOINTS:
        WAYPOINTS[town]["weather"] = condition.capitalize()
        print(f"Updated Weather Fact: WeatherAt({town}, {condition.capitalize()})")
    else:
        print(f"Error: Unknown waypoint '{town}'")

# Add dynamic operational facts
update_weather("Mau_Summit", "Fog")
add_or_update_edge("Nakuru", "Eldoret", 155, 50, "Open")  # Direct bypass fact

show_kb()

Updated Weather Fact: WeatherAt(Mau_Summit, Fog)
Added New Edge Fact: Nakuru -> Eldoret (155 km, 50t, Open)

================ TRAILER KNOWLEDGE BASE ================
Waypoint Weather Percepts:
  Nairobi      -> Weather: Clear
  Naivasha     -> Weather: Clear
  Thika        -> Weather: Clear
  Nyeri        -> Weather: Clear
  Nyahururu    -> Weather: Clear
  Gilgil       -> Weather: Clear
  Nakuru       -> Weather: Rain
  Mau_Summit   -> Weather: Fog
  Kericho      -> Weather: Clear
  Eldoret      -> Weather: Clear
  Ahero        -> Weather: Clear
  Kisumu       -> Weather: Clear

Directed Highway Edges:
  Nairobi      -> Naivasha     |  90 km | MaxLoad: 50t | Status: Open
  Nairobi      -> Thika        |  45 km | MaxLoad: 50t | Status: Open
  Thika        -> Nyeri        | 106 km | MaxLoad: 48t | Status: Open
  Nyeri        -> Nyahururu    | 101 km | MaxLoad: 45t | Status: Open
  Nyahururu    -> Gilgil       |  67 km | MaxLoad: 45t | Status: Open
  Naivasha     -> Gilgil       |  28 km

**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers** `[3 marks]`

In [3]:
class SelfDrivingTrailerKBA:
    def __init__(self, plate_no, load_tonnes):
        self.plate_no = plate_no
        self.load_tonnes = load_tonnes

    def evaluate_segment(self, src, dst):
        edge = next((e for e in DIRECTED_GRAPH.get(src, []) if e[0] == dst), None)
        if not edge:
            return {"traversable": False, "reason": f"No direct highway link from {src} to {dst}."}

        _, dist, max_w, status = edge
        weather = WAYPOINTS[src]["weather"]

        if status == "Blocked":
            return {"traversable": False, "reason": f"Segment {src} -> {dst} is BLOCKED."}
        if self.load_tonnes > max_w:
            return {"traversable": False, "reason": f"Overweight: {self.load_tonnes}t exceeds {max_w}t limit on {src} -> {dst}."}

        rule = DOMAIN_RULES.get((status, weather)) or DOMAIN_RULES.get((status, "Any"))
        if not rule:
            rule = {"action": "Proceed_Caution", "max_speed_kmh": 35, "advice": "Unmapped weather/road state: proceed with caution."}

        return {
            "traversable": True,
            "distance_km": dist,
            "weather": weather,
            "status": status,
            "action": rule["action"],
            "max_speed_kmh": rule["max_speed_kmh"],
            "advice": rule["advice"]
        }

    def find_optimal_route(self, start, goal):
        """A* Search Tree traversal using PE3 road distances g(n) and Haversine h(n)."""
        frontier = [(haversine_km(start, goal), 0, start, [start])]
        best_g = {}

        while frontier:
            f, g, current, path = heapq.heappop(frontier)
            if current == goal:
                return {"success": True, "path": path, "total_km": g}
            if current in best_g and best_g[current] <= g:
                continue
            best_g[current] = g

            for neighbor, dist, _, _ in DIRECTED_GRAPH.get(current, []):
                check = self.evaluate_segment(current, neighbor)
                if not check["traversable"]:
                    continue
                new_g = g + dist
                new_f = new_g + haversine_km(neighbor, goal)
                heapq.heappush(frontier, (new_f, new_g, neighbor, path + [neighbor]))

        return {"success": False, "reason": "No valid route satisfies trailer weight and road status constraints."}

# Step 6: Specific Problem Instance (42-tonne trailer traveling Nairobi -> Kisumu)
trailer = SelfDrivingTrailerKBA(plate_no="KDG-166T", load_tonnes=42)
result = trailer.find_optimal_route("Nairobi", "Kisumu")

# Step 7: Output Decision Plan
print(f"=== Trailer {trailer.plate_no} ({trailer.load_tonnes}t) Route & Action Plan ===")
if result["success"]:
    print("Optimal Route :", " -> ".join(result["path"]))
    print("Total Distance:", result["total_km"], "km\n")
    for i in range(len(result["path"]) - 1):
        u, v = result["path"][i], result["path"][i+1]
        step = trailer.evaluate_segment(u, v)
        print(f"{i+1}. {u} -> {v} ({step['distance_km']} km) | Weather: {step['weather']} | Action: {step['action']} (Max {step['max_speed_kmh']} km/h)")
        print(f"   Advice: {step['advice']}")
else:
    print("Routing Error:", result["reason"])

=== Trailer KDG-166T (42t) Route & Action Plan ===
Optimal Route : Nairobi -> Naivasha -> Gilgil -> Nakuru -> Mau_Summit -> Kericho -> Ahero -> Kisumu
Total Distance: 345 km

1. Nairobi -> Naivasha (90 km) | Weather: Clear | Action: Proceed_Normal (Max 80 km/h)
   Advice: Safe highway conditions: maintain cruise speed (80 km/h cap).
2. Naivasha -> Gilgil (28 km) | Weather: Clear | Action: Proceed_Normal (Max 80 km/h)
   Advice: Safe highway conditions: maintain cruise speed (80 km/h cap).
3. Gilgil -> Nakuru (43 km) | Weather: Clear | Action: Proceed_Normal (Max 80 km/h)
   Advice: Safe highway conditions: maintain cruise speed (80 km/h cap).
4. Nakuru -> Mau_Summit (52 km) | Weather: Rain | Action: Reduce_Speed_Increase_Headway (Max 55 km/h)
   Advice: Wet tarmac: engage engine retarder and double following distance.
5. Mau_Summit -> Kericho (48 km) | Weather: Fog | Action: Reduce_Speed_Increase_Headway (Max 40 km/h)
   Advice: Rift Valley/highland fog: activate fog lamps and lane-kee

**Step 8: Querying the Knowledge Base**

In [4]:
# Query 1: Inspect segment advice at foggy Mau_Summit -> Kericho
print("Query 1 (Mau_Summit -> Kericho):", trailer.evaluate_segment("Mau_Summit", "Kericho"))

# Query 2: Simulate landslide blocking Mau_Summit -> Kericho and test dynamic rerouting
print("\n[Alert] Landslide blocks Mau_Summit -> Kericho! Updating KB...")
add_or_update_edge("Mau_Summit", "Kericho", 48, 48, "Blocked")
print("Query 2 (Rerouted Nairobi -> Kisumu):", trailer.find_optimal_route("Nairobi", "Kisumu"))

# Restore segment
add_or_update_edge("Mau_Summit", "Kericho", 48, 48, "Open")

Query 1 (Mau_Summit -> Kericho): {'traversable': True, 'distance_km': 48, 'weather': 'Fog', 'status': 'Open', 'action': 'Reduce_Speed_Increase_Headway', 'max_speed_kmh': 40, 'advice': 'Rift Valley/highland fog: activate fog lamps and lane-keep assist.'}

[Alert] Landslide blocks Mau_Summit -> Kericho! Updating KB...
Updated Edge Fact: Mau_Summit -> Kericho (48 km, 48t, Blocked)
Query 2 (Rerouted Nairobi -> Kisumu): {'success': True, 'path': ['Nairobi', 'Naivasha', 'Gilgil', 'Nakuru', 'Eldoret', 'Kisumu'], 'total_km': 434}
Updated Edge Fact: Mau_Summit -> Kericho (48 km, 48t, Open)


**Step 9: Debugging the Knowledge Base** `[2 marks]`

In [5]:
print("--- Debugging & Fault Testing ---")
# Case 1: Non-existent direct road
print("Test 1 (Disconnected towns):", trailer.evaluate_segment("Nairobi", "Kisumu"))

# Case 2: Overweight 55t cargo exceeding all bridge/road limits
overweight_trailer = SelfDrivingTrailerKBA("KDH-999Z", load_tonnes=55)
print("Test 2 (Overweight 55t):", overweight_trailer.find_optimal_route("Nairobi", "Kisumu"))

# Case 3: Unmapped weather condition ('Hail') to test fallback handling
update_weather("Naivasha", "Hail")
print("Test 3 (Unmapped 'Hail' percept):", trailer.evaluate_segment("Naivasha", "Gilgil"))
update_weather("Naivasha", "Clear")

--- Debugging & Fault Testing ---
Test 1 (Disconnected towns): {'traversable': False, 'reason': 'No direct highway link from Nairobi to Kisumu.'}
Test 2 (Overweight 55t): {'success': False, 'reason': 'No valid route satisfies trailer weight and road status constraints.'}
Updated Weather Fact: WeatherAt(Naivasha, Hail)
Test 3 (Unmapped 'Hail' percept): {'traversable': True, 'distance_km': 28, 'weather': 'Hail', 'status': 'Open', 'action': 'Proceed_Caution', 'max_speed_kmh': 35, 'advice': 'Unmapped weather/road state: proceed with caution.'}
Updated Weather Fact: WeatherAt(Naivasha, Clear)


**Step 10: Final Reflection (Challenges & Recommended Improvements)** `[3 marks]`

### Challenges Faced
1. **Combining Symbolic Rules with Graph Search:** Integrating rule-based safety checks (`MaxAxleLoad`, `Blocked`, `Weather`) directly inside the A* search expansion loop required pruning invalid edges dynamically without corrupting the closed/visited set.
2. **API Availability & Latency:** Live weather API calls can time out on mobile networks; caching default waypoint states in the Knowledge Base ensures the trailer still operates safely offline.
3. **Incomplete Percept Coverage:** As verified in Step 9 (`Hail`), unmapped sensor percepts require safe fallback rules so the agent defaults to caution rather than failing.

### Recommended Improvements
1. **First-Order Logic (FOL) & Probabilistic Inference:** Transition from flat tuple rules to First-Order Logic or Bayesian Networks to reason under sensor uncertainty (e.g., partial fog visibility).
2. **Dynamic Edge-Cost Weighting:** Adjust $g(n)$ dynamically based on road gradient (e.g., Mai Mahiu escarpment fuel burn) and live traffic congestion speeds rather than static distance alone.
3. **Fleet V2V Knowledge Sharing:** Allow trailers along the Northern Corridor to broadcast roadblock or weather percepts directly into a shared cloud Knowledge Base.